# VAE로 MNIST 손글씨 복원하고 생성하기

**데이터 준비 → 인코더/디코더 정의 → 재매개변수화 → 학습/검증 → 최종 테스트 → 복원 → 생성 → 잠재 공간 관찰 → 저장/불러오기** 순서로 진행합니다.

VAE(Variational Autoencoder)는 이미지를 잠재 변수 `z`의 **확률 분포**로 압축하고 다시 이미지로 복원합니다. 일반적인 오토인코더와 달리 인코더는 하나의 코드 대신 평균과 분산을 출력합니다. 학습이 끝나면 입력 이미지 없이도 표준 정규분포에서 `z`를 뽑아 새 이미지를 생성할 수 있습니다.

기본 설정은 `QUICK_RUN = False`로 전체 데이터를 20 epoch 학습합니다. 실행 흐름만 빠르게 확인하려면 `True`로 바꿔 2,000장으로 5 epoch 학습하세요.

튜닝 실험과 후보별 비교는 **[03_vae_tuning.ipynb](03_vae_tuning.ipynb)**에서 진행합니다. 이 노트북의 ResNet VAE 클래스 정의를 그대로 재사용하며, 각 단계에서 한 변수만 바꿉니다.

전체 데이터 확인 결과와 설정 선택 근거: [튜닝 보고서](../reports/vae_tuning_20260929.md). 학습된 최종 가중치는 `outputs/notebooks/vae/confirmation_20260929/selected.pt`에 있습니다.

## 1. 라이브러리와 실행 설정

튜닝 결과를 반영하여 `LATENT_DIM = 8`, `BETA = 2`, `HIDDEN_DIM = 800`, `BATCH_SIZE = 32`, `LEARNING_RATE = 1e-3`으로 시작합니다. 학습 전반에 β를 0→2로 증가시킵니다. 2차원 잠재 공간 그림은 생략되며, 필요하면 `LATENT_DIM = 2`로 바꿔 별도로 실험하세요.

In [ ]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

SEED = 42
QUICK_RUN = False
EPOCHS = 5 if QUICK_RUN else 20  # 빠른 확인 5회 / 전체 학습 20회
TRAIN_LIMIT = 2_000 if QUICK_RUN else 0
# 원래 학습 데이터 60,000장 중 10%를 검증용으로 분리함
# 검증용 데이터를 따로 분리하는 이유
# 학습 중 모델이 과적합되는지 확인하고, 최적의 하이퍼파라미터를 선택하기 위해 필요합니다.
# 테스트 데이터는 학습과 검증에 사용되지 않은 별도의 데이터로 모델의 최종 성능을 평가하는 데 사용됩니다.
VAL_RATIO = 0.1
# 빠른 실행에서는 분리된 검증 데이터 중 500장만 사용함; 0이면 검증 데이터 전체를 사용함
VAL_LIMIT = 500 if QUICK_RUN else 0
TEST_LIMIT = 1_000 if QUICK_RUN else 0
BATCH_SIZE = 32  # 검증 순차 튜닝 및 전체 데이터 확인으로 선택한 값
HIDDEN_DIM = 800  # 검증 순차 튜닝 및 전체 데이터 확인으로 선택한 값
LATENT_DIM = 8  # 검증 순차 튜닝 및 전체 데이터 확인으로 선택한 값
LEARNING_RATE = 0.001  # 검증 순차 튜닝 및 전체 데이터 확인으로 선택한 값
# KL 목표 가중치를 설정함; 초반에는 아래 annealing으로 더 작은 값을 사용함
BETA = 2.0  # 검증 순차 튜닝 및 전체 데이터 확인으로 선택한 값
                  # KL 손실의 비중이 커진다는 것은 잠재 공간의 분포가 정규분포에 더 가깝게 강제된다는 것을 의미합니다.
                  # 너무 크게 설정하면 재구성 손실보다 KL 손실이 지배적이 되어 모델이 제대로 학습되지 않을 수 있습니다.
                  # 이는 모델이 잠재 공간에서 충분히 다양한 표현을 학습하지 못하게 만들 수 있습니다.
                  # 결과적으로 생성된 이미지의 다양성이 줄어들 수 있습니다.
# 학습 전반 절반 동안 beta를 0에서 목표값까지 선형 증가시킴
ANNEAL_EPOCHS = max(2, int(np.ceil(EPOCHS * 0.5)))
DATA_DIR = Path('/data')
# 저장 위치는 프로젝트 루트나 notebooks 디렉토리에서 실행할 때 동일합니다.
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUTPUT_DIR = PROJECT_DIR / 'outputs' / 'notebooks' / 'vae'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 램덤 시드 고정, 고정 대상은 주로 파이썬의 random, NumPy, PyTorch입니다.
# 이들 모두 랜덤한 초기값을 가지므로 시드를 고정해야 재현 가능한 결과를 얻을 수 있습니다.
# 여기서의 초기값은 다음과 같습니다.
# - random.seed(SEED): 파이썬의 random 모듈 초기값
# - np.random.seed(SEED): NumPy의 random 모듈 초기값
# - torch.manual_seed(SEED): CPU 연산에 대한 PyTorch 초기값
# - torch.cuda.manual_seed_all(SEED): GPU 연산에 대한 PyTorch 초기값

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('PyTorch:', torch.__version__)
print('device:', DEVICE)
print('latent dimension:', LATENT_DIM)

## 2. MNIST 데이터 준비와 관찰

`ToTensor()`만 적용하여 픽셀 값을 **0~1**로 유지합니다. 아래 BCE 복원 손실은 이 범위의 정답을 받습니다. 숫자 라벨은 학습에 사용하지 않으며 시각화에만 사용합니다. 첫 실행에는 MNIST 다운로드를 위한 인터넷 연결이 필요합니다.

MNIST의 `train=True` 데이터 60,000장을 **학습 54,000장 / 검증 6,000장**으로 나눕니다. 인덱스를 한 번 섞은 뒤 서로 다른 구간을 사용하므로 같은 원본 샘플이 두 집합에 들어가지 않습니다. 시드는 분할을 재현하는 역할이며, 중복 방지는 인덱스 구간 분리로 보장합니다.

- **학습(train)**: 역전파와 가중치 갱신에 사용합니다.
- **검증(validation)**: 매 에포크 손실을 비교하고 가장 좋은 모델을 선택합니다. 가중치는 갱신하지 않습니다.
- **테스트(test)**: `train=False`의 별도 10,000장을 사용하며, 모델 선택이 끝난 뒤 최종 성능을 평가합니다.

빠른 실행에서는 먼저 같은 방식으로 분할한 뒤 학습 2,000장, 검증 500장, 테스트 1,000장으로 제한합니다. 학습 전 이미지 관찰에는 검증 배치를 사용합니다.

In [ ]:
# 데이터셋에서 일부 샘플만 가져오는 유틸리티 함수입니다.
def take_subset(dataset, limit, seed):
    if limit <= 0 or limit >= len(dataset):
        return dataset
    generator = torch.Generator().manual_seed(seed)
    indices = torch.randperm(len(dataset), generator=generator)[:limit].tolist()
    return Subset(dataset, indices)

# 데이터셋을 로드하고 일부 샘플만 사용하는 경우에 대비한 설정입니다.    
# 텐서 변환을 위한 객체를 생성합니다. MNIST 데이터셋의 이미지를 PyTorch 텐서로 변환합니다.
transform = transforms.ToTensor()
# 학습용 및 테스트용 데이터셋을 로드합니다. arg 는 데이터 디렉토리, 학습 여부, 다운로드 여부, 변환 객체를 의미합니다.
# 데이터 디렉토리가 의미하는 것은 MNIST 데이터셋이 저장될 위치입니다.
# 학습 여부는 True이면 학습용 데이터셋, False이면 테스트용 데이터셋을 의미합니다.
# 다운로드 여부는 데이터셋이 로컬에 없을 경우 인터넷에서 다운로드할지를 결정합니다.
# 변환 객체는 데이터셋의 각 샘플에 적용할 변환을 정의합니다.
full_train_dataset = datasets.MNIST(DATA_DIR, train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(DATA_DIR, train=False, download=True, transform=transform)
# 일부 샘플만 사용하도록 데이터셋을 제한합니다. limit 값이 0 이하이거나 데이터셋 크기 이상이면 전체 데이터셋을 사용합니다.
# 시드는 무작위 선택을 재현하며, 학습/검증은 아래에서 원본 인덱스를 나누어 분리합니다.
# 테스트 데이터셋에서 일부 샘플을 선택함.
# 별도의 시드로 선택 순서를 재현하며, 학습·테스트 구분은 MNIST의 train 옵션으로 이루어짐. 이 옵션은 미리 정의된 학습용/테스트용 데이터셋을 선택하는 역할을 합니다.
# 검증 비율이 유효한지 확인하여 빈 학습/검증 집합이 생기는 설정을 방지함
if not 0 < VAL_RATIO < 1:
    # 잘못된 비율이면 분할 전에 원인을 알 수 있는 오류를 발생시킴
    raise ValueError('VAL_RATIO는 0과 1 사이여야 합니다.')
# 학습/검증 분할 전용 난수 생성기를 같은 시드로 초기화함
split_generator = torch.Generator().manual_seed(SEED)
# 학습용 원본 인덱스 전체를 한 번만 섞음; 각 인덱스는 정확히 한 번 등장함
split_indices = torch.randperm(len(full_train_dataset), generator=split_generator).tolist()
# 검증용 이미지 수를 정수로 계산함; 기본값은 6,000장임
val_size = int(len(full_train_dataset) * VAL_RATIO)
# 양쪽에 적어도 한 장씩 배정되는지 확인함
if not 0 < val_size < len(full_train_dataset):
    # 비율을 정수 개수로 변환한 뒤 빈 집합이 생기면 분할을 중단함
    raise ValueError('학습과 검증 데이터가 각각 최소 한 장 필요합니다.')
# 앞쪽 인덱스는 검증 전용으로 배정함
val_dataset = Subset(full_train_dataset, split_indices[:val_size])
# 나머지 인덱스는 학습 전용으로 배정하므로 검증 인덱스와 겹치지 않음
train_dataset = Subset(full_train_dataset, split_indices[val_size:])
train_dataset = take_subset(train_dataset, TRAIN_LIMIT, SEED)
# 이미 분리된 검증 집합 안에서만 필요한 개수를 선택함
val_dataset = take_subset(val_dataset, VAL_LIMIT, SEED + 1)
# 별도 테스트 집합에서 일부를 선택함; 시드는 선택 재현용이며 집합 분리와는 무관함
test_dataset = take_subset(test_dataset, TEST_LIMIT, SEED + 2)

# 데이터로더를 생성합니다. 학습용 데이터로더는 셔플을 적용하고, 테스트용 데이터로더는 셔플을 적용하지 않습니다.
# 데이터로더 옵션을 설정합니다. 배치 크기, 워커 수, 핀 메모리 여부를 지정합니다.
# 배치 크기는 한 번에 모델에 입력되는 샘플 수를 의미합니다.
# num_workers는 데이터를 로드할 때 사용할 서브 프로세스 수를 의미합니다., 0은 메인 프로세스에서 데이터를 로드함을 의미합니다. 보통 CPU 코어 수에 맞게 설정합니다.
# pin_memory는 데이터를 CUDA 고정 메모리에 올릴지를 결정합니다. CUDA를 사용할 경우 True로 설정하면 성능이 향상될 수 있습니다.
loader_options = dict(batch_size=BATCH_SIZE, num_workers=0, pin_memory=DEVICE.type == 'cuda')
# 학습용 및 테스트용 데이터로더를 생성합니다. **는 딕셔너리 형태의 추가 옵션을 언팩하여 전달하는 역할을 합니다.
train_loader = DataLoader(train_dataset, shuffle=True, **loader_options)
# 검증 데이터는 순서를 고정하여 매 에포크 같은 순서로 평가함
# 비교실험과 동일하게 검증 batch는 학습 batch와 무관하게 128로 고정함
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=0)
# 테스트 데이터도 순서를 고정함
test_loader = DataLoader(test_dataset, shuffle=False, **loader_options)

# 검증용 데이터로더에서 첫 번째 배치를 가져와 이미지와 레이블을 확인합니다.
# 시각적으로 첫 번째 배치의 이미지를 확인합니다.
images, labels = next(iter(val_loader))
print('train/val/test:', len(train_dataset), '/', len(val_dataset), '/', len(test_dataset))
print('image batch:', tuple(images.shape))
fig, axes = plt.subplots(1, 10, figsize=(12, 2))
for axis, image, label in zip(axes, images[:10], labels[:10]):
    axis.imshow(image.squeeze(0), cmap='gray', vmin=0, vmax=1)
    axis.set_title(str(label.item()))
    axis.axis('off')
plt.tight_layout()
plt.show()

## 3. VAE 모델과 재매개변수화

- **ResNet 인코더**: `(B, 1, 28, 28)` → `(B, 32, 28, 28)` → `(B, 64, 14, 14)` → `(B, 128, 7, 7)` → 전역 평균 pooling → 평균 `mu`와 로그 분산 `logvar`, 각각 `(B, LATENT_DIM)`.
- **잔차 블록**: 두 합성곱의 결과 `F(x)`에 shortcut을 더해 `ReLU(F(x) + shortcut(x))`를 계산합니다. 크기가 달라지는 블록은 1×1 합성곱으로 shortcut의 크기를 맞춥니다. 흑백 28×28 이미지에 맞춘 소형 ResNet이며 사전 학습 가중치 없이 학습합니다. BatchNorm은 학습 중 배치 통계를, 평가 중 저장된 통계를 사용합니다.
- **재매개변수화**: $z = \mu + \exp(0.5\,\mathrm{logvar})\,\epsilon$, $\epsilon \sim \mathcal{N}(0,I)$.
- **디코더**: `z` → 784개 픽셀의 logit. 표시할 때 sigmoid를 적용해 0~1로 바꿉니다.

`logvar`는 표준편차가 아니라 **분산의 로그**입니다. 난수 `epsilon`과 학습 가능한 평균·표준편차를 분리하면 샘플링을 포함하면서도 인코더까지 기울기를 전달할 수 있습니다. `sample=False`이면 `z = mu`로 복원하여 같은 입력의 결과를 비교하기 쉽습니다. `eval()`만으로 샘플링이 꺼지지는 않습니다.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        # Residual block의 본체를 정의합니다. Conv-BN-ReLU-Conv-BN 구조를 가집니다.
        self.residual = nn.Sequential(
            # 첫 번째 Conv-BN-ReLU 레이어입니다. 입력 채널 수와 출력 채널 수, 스트라이드를 지정합니다. 패딩은 1로 설정하여 출력 크기를 유지합니다.
            # 피쳐맵을 유지하면서 채널 수를 변경할 수 있습니다.
            nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False),
            # 배치 정규화 레이어입니다. 출력 채널 수를 지정합니다.
            # 이를 통해 가중치가 너무 커지는 것을 방지하고 학습을 안정화할 수 있습니다.
            # 내부적으론 미니배치의 평균과 분산을 계산하여 정규화합니다.
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
        )
        # 해상도나 채널 수가 달라지면 shortcut도 같은 shape로 변환합니다.
        self.shortcut = nn.Identity()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        self.activation = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.activation(self.residual(x) + self.shortcut(x))


class VAE(nn.Module):
    def __init__(self, latent_dim=2, hidden_dim=400):
        super().__init__()
        # MNIST용 소형 ResNet: 3개 stage마다 잔차 블록 2개씩 사용합니다.
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1, bias=False),  # (B, 32, 28, 28)
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            ResidualBlock(32, 32),
            ResidualBlock(32, 32),
            ResidualBlock(32, 64, stride=2),           # (B, 64, 14, 14)
            ResidualBlock(64, 64),
            ResidualBlock(64, 128, stride=2),          # (B, 128, 7, 7)
            ResidualBlock(128, 128),
            nn.AdaptiveAvgPool2d(1),                  # (B, 128, 1, 1)
            nn.Flatten(),
            nn.Linear(128, hidden_dim),
            nn.ReLU(),
        )
        # 잠재 공간에서 샘플링된 벡터를 입력으로 받아 이미지를 재구성하는 디코더를 정의합니다.
        # 잠재 공간의 평균과 로그 분산을 계산하는 fully connected 레이어를 정의합니다.
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, 784),
            # 최종 출력은 28x28 크기의 이미지로 재구성됩니다. sigmoid는 forward에서 적용할 수 있습니다.
            # 레이어가 하나인데 출력 크기를 28x28로 맞추기 위해 reshape가 필요합니다.
        )

    # 인코더, 디코더, reparameterization을 포함한 VAE의 전체 구조를 정의합니다.
    # 인코더의 인풋을 넣으면 평균과 분산을 계산하여 잠재 벡터를 샘플링합니다.
    def encode(self, x):
        hidden = self.encoder(x)
        return self.fc_mu(hidden), self.fc_logvar(hidden)

    # reparameterization trick을 통해 잠재 벡터를 샘플링합니다. 학습 시에는 평균과 분산을 이용해 샘플링하고, 평가 시에는 평균을 사용합니다.
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        epsilon = torch.randn_like(std)
        return mu + std * epsilon

    # 디코더는 잠재 벡터를 입력으로 받아 원래 이미지 크기로 재구성합니다.

    def decode(self, z):
        return self.decoder(z).reshape(-1, 1, 28, 28)  # sigmoid 적용 전 logit
    # 인코더를 통해 평균과 분산을 계산하고, reparameterization trick을 통해 잠재 벡터를 샘플링한 후 디코더를 통해 이미지를 재구성합니다.
    def forward(self, x, sample=True):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar) if sample else mu
        return self.decode(z), mu, logvar

model = VAE(LATENT_DIM, HIDDEN_DIM).to(DEVICE)
model.eval()  # shape 확인 시 BatchNorm 통계를 갱신하지 않습니다.
with torch.no_grad():
    logits, mu, logvar = model(images[:4].to(DEVICE))
print(model)
print('input:', tuple(images[:4].shape))
print('mu / logvar:', tuple(mu.shape), '/', tuple(logvar.shape))
print('reconstruction logits:', tuple(logits.shape))
print('parameters:', sum(p.numel() for p in model.parameters()))

## 4. 복원 손실과 KL divergence

학습 손실은 `reconstruction + BETA * KL`입니다.

**복원 손실**은 입력과 출력의 픽셀별 BCE입니다. `binary_cross_entropy_with_logits`는 sigmoid와 BCE를 수치적으로 안정하게 함께 계산하므로, 손실 계산 전에 sigmoid를 적용하지 않습니다. 여기서는 0~1의 회색조 값을 BCE의 soft target으로 사용합니다.

**KL 손실**은 인코더의 대각 정규분포 $q(z|x)$가 사전 분포 $\mathcal{N}(0,I)$에서 얼마나 벗어났는지 측정합니다:

$$D_{KL} = -\frac12 \sum_j (1 + \mathrm{logvar}_j - \mu_j^2 - \exp(\mathrm{logvar}_j)).$$

픽셀/잠재 차원별 손실은 **합산**, 배치별 손실은 **평균**합니다. 따라서 BCE와 KL의 상대적인 크기가 배치 크기에 따라 달라지지 않습니다. `BETA = 1`일 때 표준 VAE의 음의 ELBO를 샘플링으로 추정합니다.

In [ ]:
def vae_loss(logits, targets, mu, logvar, beta=1.0):
    reconstruction = F.binary_cross_entropy_with_logits(
        logits, targets, reduction='none',
    ).flatten(start_dim=1).sum(dim=1).mean()
    kl = -0.5 * (1 + logvar - mu.square() - logvar.exp()).sum(dim=1).mean()
    return reconstruction + beta * kl, reconstruction, kl

## 5. 학습과 검증으로 모델 선택

이전 예제는 흐름을 단순하게 구성하면서 매 에포크 테스트 데이터를 평가했습니다. 이제 별도 검증 집합을 사용해 학습 진행과 과적합을 관찰하고 **목표 β에 도달한 이후 검증 BCE+KL(고정 가중치 1)이 가장 낮은 에포크의 모델**을 선택합니다. 테스트 데이터는 모델 선택에 사용하지 않습니다.

학습에서는 입력 이미지 자체가 복원의 정답입니다. 검증에서는 `eval()` 동작과 기울기 기록 비활성화로 가중치 및 BatchNorm 통계가 변하지 않습니다. 평가에서도 `z`를 샘플링하므로 검증에는 고정 난수 스트림을 사용하여 샘플링 변동을 줄입니다. 손실은 이미지 한 장당 평균이며, 학습 손실은 현재 β를 반영하고, 모델 선택은 고정 가중치 1의 BCE+KL로 비교합니다. 설정한 epoch를 모두 학습한 뒤 최적 에포크의 가중치를 복원합니다.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
history = []
# 지금까지 가장 낮은 검증 손실을 저장함; 첫 평가가 선택되도록 무한대로 시작함
best_val_reference = float('inf')
# 가장 좋은 모델이 나온 에포크 번호를 기록함
best_epoch = 0
# 최적 모델의 가중치와 BatchNorm 통계를 보관할 변수를 준비함
best_model_state = None

# 옵티마이저를 정의하고 학습 기록을 저장할 리스트를 초기화합니다.
# 학습을 수행하는 함수를 정의합니다. 학습 모드와 평가 모드를 구분하고, 손실을 계산하며, 옵티마이저를 사용하여 가중치를 업데이트합니다.
def run_epoch(model, loader, optimizer=None, beta=None):
    # beta가 지정되지 않은 최종 평가에서는 목표 가중치를 사용함
    beta = BETA if beta is None else beta
    # 검증용 난수는 매번 같은 값으로 초기화하며 학습 난수와 분리함
    eval_generator = torch.Generator(device=DEVICE).manual_seed(1042)
    # 학습 모드인지 평가 모드인지를 결정합니다. optimizer가 None이 아니면 학습 모드입니다.
    training = optimizer is not None
    # 모델을 학습 모드 또는 평가 모드로 설정합니다.
    model.train(training)
    # 손실 합계와 샘플 수를 초기화합니다. totals는 손실, 재구성 손실, KL 발산을 저장합니다. count는 처리한 샘플 수를 저장합니다.
    totals = np.zeros(3, dtype=np.float64)
    count = 0
    # gradient 계산을 활성화하거나 비활성화합니다. 학습 모드일 때만 gradient를 계산합니다.
    # with 블록은 학습과 평가 모두 실행됩니다. training 값에 따라 기울기 기록만 켜거나 끕니다.
    with torch.set_grad_enabled(training):
        # 데이터 로더에서 배치를 가져와 모델에 입력하고 손실을 계산합니다. 학습 모드일 때는 역전파를 수행하고 옵티마이저를 통해 가중치를 업데이트합니다.
        for batch_images, _ in loader:
            batch_images = batch_images.to(DEVICE)
            if training:
                # 옵티마이저의 gradient를 초기화합니다. set_to_none=True로 설정하면 메모리 사용을 줄일 수 있습니다.
                optimizer.zero_grad(set_to_none=True)
            # 옵티마이저를 초기화한 후 모델에 입력을 전달하여 출력과 손실을 계산합니다.
            # 학습에서는 매번 샘플링하고 검증에서는 고정된 epsilon을 사용함
            if training:
                # 인코더와 디코더를 통과하여 학습용 확률적 복원을 구함
                logits, mu, logvar = model(batch_images)
            else:
                # 검증 이미지의 잠재 분포를 계산함
                mu, logvar = model.encode(batch_images)
                # 검증 배치의 shape에 맞춰 고정 난수 스트림에서 epsilon을 생성함
                epsilon = torch.randn(mu.shape, device=DEVICE, generator=eval_generator)
                # 같은 검증 난수로 재매개변수화한 뒤 디코딩함
                logits = model.decode(mu + (0.5 * logvar).exp() * epsilon)
            # 모델의 출력과 손실을 계산합니다. logits는 재구성된 이미지, mu와 logvar는 잠재 벡터의 평균과 로그 분산입니다.
            loss, reconstruction, kl = vae_loss(logits, batch_images, mu, logvar, beta)
            # 학습 모드일 때는 역전파를 수행하고 옵티마이저를 통해 가중치를 업데이트합니다.
            if training:
                # 손실을 역전파합니다.
                loss.backward()
                # 옵티마이저를 통해 가중치를 업데이트합니다.
                optimizer.step()
            # 배치 크기를 계산하고 손실 합계와 샘플 수를 업데이트합니다.
            batch_size = batch_images.size(0)
            # 배치 손실을 계산하여 totals에 누적합니다. batch_size를 곱하여 배치 크기에 따른 가중치를 반영합니다.
            totals += np.array([loss.item(), reconstruction.item(), kl.item()]) * batch_size
            # 처리한 샘플 수를 업데이트합니다.
            count += batch_size
    # 에폭이 끝난 후 평균 손실을 계산하여 반환합니다. totals를 count로 나누어 배치 크기에 따른 가중치를 반영한 평균 손실을 구합니다.
    return dict(zip(('loss', 'reconstruction', 'kl'), (totals / count).tolist()))

for epoch in range(1, EPOCHS + 1):
    # 첫 epoch는 0, warmup 마지막 epoch부터 목표 BETA를 사용함
    effective_beta = BETA * min(1.0, (epoch - 1) / (ANNEAL_EPOCHS - 1))
    # 이번 epoch의 KL 가중치로 학습함
    train_metrics = run_epoch(model, train_loader, optimizer, beta=effective_beta)
    val_metrics = run_epoch(model, val_loader, beta=effective_beta)
    history.append({'epoch': epoch, 'effective_beta': effective_beta, 'train': train_metrics, 'val': val_metrics})
    print(
        f"epoch={epoch:02d} beta={effective_beta:.3f} train={train_metrics['loss']:.2f} "
        f"val={val_metrics['loss']:.2f} "
        f"val reconstruction={val_metrics['reconstruction']:.2f} "
        f"val KL={val_metrics['kl']:.2f}"
    )
    # 현재 검증 전체 손실이 이전 최솟값보다 낮으면 최적 모델을 교체함
    # beta 변화에 영향받지 않는 공통 BCE+KL 기준을 계산함
    val_reference = val_metrics['reconstruction'] + val_metrics['kl']
    # 목표 beta에 도달한 모델 중 공통 검증 점수가 가장 낮은 모델만 선택함
    if effective_beta == BETA and val_reference < best_val_reference:
        # 새로운 최소 검증 손실을 Python 숫자로 기록함
        best_val_reference = val_reference
        # 해당 모델이 나온 에포크를 기록함
        best_epoch = epoch
        # CPU에 독립된 복사본을 보관하여 이후 학습이 저장된 최적 가중치를 바꾸지 않게 함
        best_model_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}

# 유효한 검증 결과가 없으면 잘못된 모델을 최종 평가하지 않도록 중단함
if best_model_state is None:
    # 에포크 수나 비정상 손실 등의 설정/학습 문제를 확인할 수 있도록 오류를 발생시킴
    raise RuntimeError('선택된 모델이 없습니다. EPOCHS와 검증 손실을 확인하세요.')
# 모든 에포크가 끝난 뒤 검증 손실이 가장 낮았던 가중치와 BatchNorm 통계를 복원함
model.load_state_dict(best_model_state)
# 최종 평가와 이미지 생성에 사용할 모델을 평가 모드로 설정함
model.eval()
# 최적 모델이 선택된 시점과 기준이 된 검증 손실을 출력함
print(f'best epoch={best_epoch:02d}, val BCE+KL={best_val_reference:.2f}')

### 5-1. 선택된 모델의 최종 테스트

검증으로 선택한 모델을 별도의 테스트 집합에서 한 번 평가합니다. 이 지표는 모델 선택이나 가중치 업데이트에 사용하지 않습니다. 아래 복원·생성·저장도 같은 최적 모델을 사용합니다.

In [ ]:
# 학습과 모델 선택이 끝난 뒤 테스트 전체를 한 번 순회함; optimizer가 없어 가중치는 갱신하지 않음
test_metrics = run_epoch(model, test_loader)
# 최종 모델의 테스트 전체 손실, 복원 손실과 KL 손실을 출력함
print(
    # 검증으로 선택한 모델의 테스트 전체 손실을 표시함
    f"final test={test_metrics['loss']:.2f} "
    # 원본 픽셀을 얼마나 잘 복원하는지 별도로 표시함
    f"reconstruction={test_metrics['reconstruction']:.2f} "
    # 잠재 분포가 표준 정규분포에서 얼마나 벗어나는지 표시함
    f"KL={test_metrics['kl']:.2f}"
)

## 6. 학습 곡선과 이미지 복원

위 행은 원본, 아래 행은 잠재 평균 `mu`를 디코딩한 복원입니다. 빠른 실행에서는 흐릿한 결과가 정상일 수 있습니다. 복원 손실과 KL 손실을 따로 관찰하세요.

곡선은 학습과 검증 손실입니다. 복원에는 검증 이미지와 검증 손실로 선택한 최적 모델을 사용합니다.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3))
for axis, metric in zip(axes, ('loss', 'reconstruction', 'kl')):
    for split in ('train', 'val'):
        axis.plot([row['epoch'] for row in history],
                  [row[split][metric] for row in history], marker='o', label=split)
    axis.set(xlabel='epoch', ylabel='per image', title=metric)
    axis.legend()
plt.tight_layout()
plt.show()

model.eval()
sample_images = images[:10].to(DEVICE)
with torch.inference_mode():
    reconstructed = model(sample_images, sample=False)[0].sigmoid().cpu()
fig, axes = plt.subplots(2, 10, figsize=(12, 3))
for index in range(10):
    for row, image in enumerate((images[index], reconstructed[index])):
        axes[row, index].imshow(image.squeeze(0), cmap='gray', vmin=0, vmax=1)
        axes[row, index].axis('off')
axes[0, 0].set_title('Original')
axes[1, 0].set_title('Reconstruction')
plt.tight_layout()
plt.show()

## 7. 입력 없이 새 이미지 생성

표준 정규분포에서 `z`를 뽑아 디코더에 넣습니다. 여기서 인코더나 입력 이미지는 사용하지 않습니다. 표시하는 값은 디코더가 예측한 픽셀 확률이며, 픽셀 자체를 다시 Bernoulli 샘플링하지는 않습니다.

In [ ]:
model.eval()
with torch.inference_mode():
    # 잠재 공간에서 샘플을 생성합니다. z는 표준 정규 분포에서 샘플링한 잠재 벡터입니다.
    # 랜덤하게 샘플을 생성합니다.
    z = torch.randn(16, LATENT_DIM, device=DEVICE)
    generated = model.decode(z).sigmoid().cpu()
fig, axes = plt.subplots(4, 4, figsize=(5, 5))
# 생성된 샘플을 4x4 격자로 시각화합니다.
for axis, image in zip(axes.flat, generated):
    axis.imshow(image.squeeze(0), cmap='gray', vmin=0, vmax=1)
    axis.axis('off')
plt.suptitle('Samples from N(0, I)')
plt.tight_layout()
plt.show()

## 8. 2차원 잠재 공간 관찰

왼쪽은 검증 이미지의 `mu`를 숫자 라벨로 색칠한 그림입니다. 라벨은 학습에 사용하지 않았으므로 숫자별 분리가 보장되지는 않습니다. 오른쪽은 잠재 좌표를 격자로 움직이며 생성한 이미지입니다. 인접 좌표에서 모양이 어떻게 변하는지 관찰하세요. `LATENT_DIM`을 2 이외의 값으로 바꾸면 이 셀의 시각화는 건너뜁니다.

In [ ]:
if LATENT_DIM == 2:
    means, label_batches = [], []
    model.eval()
    with torch.inference_mode():
        for batch_images, batch_labels in val_loader:
            mu, _ = model.encode(batch_images.to(DEVICE))
            means.append(mu.cpu())
            label_batches.append(batch_labels)
        coordinates = torch.cat(means).numpy()
        colors = torch.cat(label_batches).numpy()
        grid_size = 15
        grid_x = torch.linspace(-2, 2, grid_size)
        grid_y = torch.linspace(2, -2, grid_size)
        yy, xx = torch.meshgrid(grid_y, grid_x, indexing='ij')
        grid_z = torch.stack((xx.flatten(), yy.flatten()), dim=1).to(DEVICE)
        tiles = model.decode(grid_z).sigmoid().cpu().reshape(grid_size, grid_size, 28, 28)
        canvas = tiles.permute(0, 2, 1, 3).reshape(grid_size * 28, grid_size * 28)
    fig, axes = plt.subplots(1, 2, figsize=(13, 6))
    scatter = axes[0].scatter(coordinates[:, 0], coordinates[:, 1], c=colors,
                              cmap='tab10', vmin=-0.5, vmax=9.5, s=5, alpha=0.7)
    fig.colorbar(scatter, ax=axes[0], ticks=range(10), label='digit label')
    axes[0].set(xlabel='mu[0]', ylabel='mu[1]', title='Encoded validation images')
    axes[1].imshow(canvas, cmap='gray', vmin=0, vmax=1, extent=(-2, 2, -2, 2))
    axes[1].set(xlabel='z[0]', ylabel='z[1]', title='Decoded latent grid')
    plt.tight_layout()
    plt.show()
else:
    print('잠재 공간 시각화는 LATENT_DIM = 2일 때 실행됩니다.')

## 9. 모델 저장

가중치, 모델 크기, 학습 설정과 손실 기록을 저장합니다. 저장 파일은 Git 관리에서 제외됩니다.

ResNet 모델은 `mnist_resnet_vae_notebook.pt`에 저장합니다. 기존 Linear 인코더의 체크포인트와 구조가 다르므로 새로 학습한 가중치를 사용하세요.

저장되는 가중치는 마지막 에포크가 아니라 **목표 β에 도달한 이후 공통 검증 BCE+KL이 가장 낮았던 에포크**의 모델입니다. 선택된 에포크, 최저 검증 손실과 최종 테스트 지표도 함께 저장합니다.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
checkpoint_path = OUTPUT_DIR / 'mnist_resnet_vae_notebook.pt'
torch.save({
    'model_state': model.state_dict(),
    'config': {
        'encoder': 'small_resnet', 'latent_dim': LATENT_DIM, 'hidden_dim': HIDDEN_DIM,
        'epochs': EPOCHS, 'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE, 'beta': BETA, 'seed': SEED,
        'train_limit': TRAIN_LIMIT, 'test_limit': TEST_LIMIT,
        # 동일한 검증 집합을 재현할 수 있도록 분할 비율과 검증 데이터 제한을 저장함
        'val_ratio': VAL_RATIO, 'val_limit': VAL_LIMIT,
        # KL 가중치 증가 기간과 모델 선택 기준을 저장함
        'anneal_epochs': ANNEAL_EPOCHS, 'selection_metric': 'validation BCE + KL',
    },
    'history': history,
    # 저장된 가중치가 어느 에포크에서 선택됐는지 기록함
    'best_epoch': best_epoch,
    # 모델 선택 기준이 된 최저 검증 손실을 저장함
    'best_val_reference': best_val_reference,
    # 모델 선택이 끝난 뒤 측정한 테스트 손실을 저장함
    'test_metrics': test_metrics,
}, checkpoint_path)
print('saved:', checkpoint_path)

## 10. 저장된 모델을 불러와 생성하기

커널을 다시 시작했다면 **1번 설정과 3번 모델 정의** 코드 셀을 먼저 실행하세요. 3번의 shape 확인에는 데이터가 필요하므로 해당 셀까지 통째로 실행하려면 2번도 실행하세요. 클래스 정의만 실행했다면 데이터 다운로드 없이 아래 셀로 생성할 수 있습니다. 기존 저장 파일이 필요합니다.

In [ ]:
checkpoint_path = OUTPUT_DIR / 'mnist_resnet_vae_notebook.pt'
checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
config = checkpoint['config']
loaded_model = VAE(config['latent_dim'], config['hidden_dim'])
loaded_model.load_state_dict(checkpoint['model_state'])
loaded_model = loaded_model.to(DEVICE).eval()
with torch.inference_mode():
    z = torch.randn(10, config['latent_dim'], device=DEVICE)
    loaded_samples = loaded_model.decode(z).sigmoid().cpu()
fig, axes = plt.subplots(1, 10, figsize=(12, 2))
for axis, image in zip(axes, loaded_samples):
    axis.imshow(image.squeeze(0), cmap='gray', vmin=0, vmax=1)
    axis.axis('off')
plt.tight_layout()
plt.show()

## 다음 실험

1. `QUICK_RUN = False`로 바꿔 충분히 학습하고 복원과 생성 품질을 비교하세요.
2. `LATENT_DIM`을 2, 16, 32로 바꿔 복원 손실과 생성 이미지를 비교하세요.
3. `BETA`를 0.1, 1, 4로 바꿔 복원과 잠재 분포 제약 사이의 변화를 관찰하세요. 1이 아닌 경우 beta-VAE 목적 함수를 사용합니다.
4. 같은 입력에 대해 `sample=True`와 `sample=False`의 복원을 여러 번 비교하세요.
5. `BETA = 0`일 때 복원이 좋아져도 표준 정규분포에서 생성한 결과가 나빠질 수 있는 이유를 생각해 보세요.

설정 비교에는 검증 손실과 검증 이미지의 복원을 사용하세요. 최종 테스트 결과를 보고 설정을 반복해서 고르면 테스트 집합도 모델 선택에 간접적으로 사용하게 됩니다.